# House Price Prediction using Multiple Linear Regression

This notebook builds a complete **Multiple Linear Regression** model for house-price prediction using the uploaded dataset.

### Workflow
1. Import libraries
2. Load and inspect the dataset
3. Data cleaning and preprocessing
4. Exploratory Data Analysis (EDA)
5. Correlation heatmap
6. Prepare features and target
7. Train/test split
8. Train Multiple Linear Regression
9. Evaluate the model
10. Actual vs. predicted plot
11. Residual analysis
12. Feature coefficients
13. House-price prediction function


In [ ]:
# 1. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["axes.grid"] = True


In [ ]:
# 2. Load dataset
DATA_PATH = r"/mnt/data/data(1).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# 3. Basic dataset information
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# 4. Descriptive statistics
display(df.describe(include="all").T)


## 5. Data preprocessing

The dataset contains:
- `price` as the prediction target.
- Numerical house/property variables.
- `city` and `statezip` as categorical variables.
- `date`, from which year and month are extracted.
- `street` and `country`, which are not useful for this regression setup and are removed.

Categorical variables are one-hot encoded inside a scikit-learn pipeline, while numerical missing values are median-imputed and scaled.


In [ ]:
# 5. Clean and engineer date features
data = df.copy()

# Convert date to datetime where possible
if "date" in data.columns:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data["sale_year"] = data["date"].dt.year
    data["sale_month"] = data["date"].dt.month
    data.drop(columns=["date"], inplace=True)

# Drop identifier/high-cardinality or constant location columns
drop_cols = [c for c in ["street", "country"] if c in data.columns]
data.drop(columns=drop_cols, inplace=True)

# Make sure target is numeric
data["price"] = pd.to_numeric(data["price"], errors="coerce")
data = data.dropna(subset=["price"])

print("Shape after preprocessing:", data.shape)
display(data.head())


## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
plt.figure(figsize=(9, 5))
plt.hist(data["price"], bins=50)
plt.xlabel("House Price")
plt.ylabel("Frequency")
plt.title("Distribution of House Prices")
plt.tight_layout()
plt.show()


In [ ]:
# Numeric feature distributions
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()

data[numeric_cols].hist(bins=30, figsize=(16, 14))
plt.suptitle("Numeric Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Price vs. important numerical features
candidate_features = [
    c for c in ["sqft_living", "sqft_lot", "bedrooms", "bathrooms",
                "floors", "sqft_above", "sqft_basement", "yr_built"]
    if c in data.columns
]

for feature in candidate_features:
    plt.figure(figsize=(8, 5))
    plt.scatter(data[feature], data["price"], alpha=0.45)
    plt.xlabel(feature)
    plt.ylabel("Price")
    plt.title(f"House Price vs {feature}")
    plt.tight_layout()
    plt.show()


## 7. Correlation Heatmap

Correlation helps identify linear relationships between numerical variables. It does **not** prove causation.


In [ ]:
# Correlation matrix
corr = data.select_dtypes(include=np.number).corr()

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Heatmap")

fig.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

print("Correlation with price:")
display(
    corr["price"]
    .sort_values(ascending=False)
    .to_frame("correlation_with_price")
)


## 8. Prepare Features and Target

In [ ]:
# Separate target and predictors
X = data.drop(columns=["price"])
y = data["price"]

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)
print("\nNumber of predictors before encoding:", X.shape[1])


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 9. Build and Train the Multiple Linear Regression Model

The preprocessing and regression model are combined into one pipeline so the same transformations are automatically applied during training and prediction.


In [ ]:
# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Multiple Linear Regression
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model training completed successfully.")


## 10. Model Evaluation

In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluation helper
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

train_mae, train_rmse, train_r2 = regression_metrics(y_train, y_train_pred)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, y_test_pred)

results = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "MAE": [train_mae, test_mae],
    "RMSE": [train_rmse, test_rmse],
    "R2 Score": [train_r2, test_r2]
})

display(results)


In [ ]:
# Actual vs. Predicted prices
plt.figure(figsize=(8, 7))
plt.scatter(y_test, y_test_pred, alpha=0.55)

min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs. Predicted House Prices")
plt.tight_layout()
plt.show()


In [ ]:
# Residual analysis
residuals = y_test - y_test_pred

plt.figure(figsize=(9, 5))
plt.scatter(y_test_pred, residuals, alpha=0.55)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs. Predicted Prices")
plt.tight_layout()
plt.show()


In [ ]:
# Prediction error distribution
plt.figure(figsize=(9, 5))
plt.hist(residuals, bins=50)
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")
plt.tight_layout()
plt.show()


## 11. Regression Coefficients

The coefficients show the direction and magnitude of each encoded feature's contribution after preprocessing. Because numerical variables are standardized, their coefficients are on a standardized scale.


In [ ]:
# Extract feature names and coefficients
preprocessor_fitted = model.named_steps["preprocessor"]
regressor = model.named_steps["regressor"]

feature_names = preprocessor_fitted.get_feature_names_out()
coefficients = regressor.coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_df["absolute_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("absolute_coefficient", ascending=False)

display(coef_df.head(20))


In [ ]:
# Plot the 15 largest coefficients by absolute magnitude
top_coef = coef_df.head(15).sort_values("coefficient")

plt.figure(figsize=(10, 7))
plt.barh(top_coef["feature"], top_coef["coefficient"])
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Top 15 Regression Coefficients")
plt.tight_layout()
plt.show()


## 12. House Price Prediction Function

Use the function below to enter the characteristics of a new house. It uses the exact same preprocessing pipeline learned during training.


In [ ]:
def predict_house_price(
    bedrooms,
    bathrooms,
    sqft_living,
    sqft_lot,
    floors,
    waterfront=0,
    view=0,
    condition=3,
    sqft_above=None,
    sqft_basement=0,
    yr_built=2000,
    yr_renovated=0,
    city=None,
    statezip=None,
    sale_year=None,
    sale_month=None
):
    # Defaults for sqft_above and date features
    if sqft_above is None:
        sqft_above = sqft_living

    if sale_year is None:
        sale_year = int(data["sale_year"].median()) if "sale_year" in data.columns else 2014

    if sale_month is None:
        sale_month = int(data["sale_month"].median()) if "sale_month" in data.columns else 6

    # Create one-row DataFrame with the same feature columns used for training
    row = {}

    for col in X.columns:
        row[col] = np.nan

    values = {
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "sqft_living": sqft_living,
        "sqft_lot": sqft_lot,
        "floors": floors,
        "waterfront": waterfront,
        "view": view,
        "condition": condition,
        "sqft_above": sqft_above,
        "sqft_basement": sqft_basement,
        "yr_built": yr_built,
        "yr_renovated": yr_renovated,
        "city": city,
        "statezip": statezip,
        "sale_year": sale_year,
        "sale_month": sale_month
    }

    for col, value in values.items():
        if col in row:
            row[col] = value

    input_df = pd.DataFrame([row], columns=X.columns)

    prediction = model.predict(input_df)[0]
    return prediction


# Example prediction
example_price = predict_house_price(
    bedrooms=3,
    bathrooms=2,
    sqft_living=1800,
    sqft_lot=5000,
    floors=1,
    waterfront=0,
    view=0,
    condition=3,
    sqft_above=1600,
    sqft_basement=200,
    yr_built=2000,
    yr_renovated=0,
    city=None,
    statezip=None
)

print(f"Predicted house price: ${example_price:,.2f}")


## 13. Important Notes

- The model is a **Multiple Linear Regression** model.
- `city` and `statezip` are handled using one-hot encoding.
- Numerical variables are standardized.
- The prediction function accepts `None` for categorical variables; the preprocessing pipeline handles missing categorical values using the most frequent category.
- A low R² score means the linear model does not explain a large portion of the price variation in this dataset. This is useful information rather than a coding error.
- For better predictive performance, nonlinear models such as Random Forest, Gradient Boosting, or XGBoost can be compared against this baseline.
